<a href="https://colab.research.google.com/github/bsheese/cs377/blob/18_classification_redesign/18_classification/18_1_Classification_Basics/18_1_3_Model_Selection_Multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Selection & Introduction to Multiclass

**Author:** Brad Sheese

---

## Introduction

Part A: How do we choose between competing models? (AIC, BIC, Cross-Validation)
Part B: What happens when we have MORE than two classes?

### Learning Objectives
By the end of this notebook, you will be able to:
1. Use AIC and BIC for model selection
2. Apply cross-validation for robust evaluation
3. Understand multiclass classification strategies
4. Compare One-vs-Rest vs. Softmax approaches

## Part A: Model Selection with AIC/BIC

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load credit default data
data = fetch_openml(name='credit-g', version=1, as_frame=True)
df = data.frame
y = (df['class'] == 'bad').astype(int)
X = df.drop(columns=['class'])
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# We'll compare 3 model types
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree (depth=3)': DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
}

print("Comparing 3 model types...")

## AIC and BIC

In [ ]:
from sklearn.metrics import accuracy_score

# Compare models using 5-fold cross-validation
cv_results = {}

for name, model in models.items():
    if name == 'Logistic Regression':
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    else:
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    
    cv_results[name] = cv_scores
    print(f"{name}:")
    print(f"  CV Accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

plt.figure(figsize=(10, 5))
names = list(cv_results.keys())
means = [cv_results[n].mean() for n in names]
stds = [cv_results[n].std() for n in names]

bars = plt.bar(names, means, yerr=stds, color=['steelblue', 'orange', 'green'], alpha=0.8)
plt.ylabel('Accuracy')
plt.title('5-Fold Cross-Validation: Model Comparison')
plt.ylim(0.6, 0.9)
for bar, mean, std in zip(bars, means, stds):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.01, f'{mean:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Part B: Multiclass Classification

What if we need to predict MORE than 2 classes?

Examples: Digit recognition (0-9), Customer segments (low/medium/high)

In [ ]:
# Use wine dataset for multiclass (encoded version)
from sklearn.datasets import load_wine
wine = load_wine()
X_wine = wine.data  # Already numeric features
y_wine = wine.target  # 0, 1, 2 (three classes)

print("Wine Dataset (multiclass):")
print(f"Classes: {np.unique(y_wine)}")
print(f"Class distribution: {np.bincount(y_wine)}")

X_wine_train, X_wine_test, y_wine_train, y_wine_test = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

scaler_wine = StandardScaler()
X_wine_train_scaled = scaler_wine.fit_transform(X_wine_train)
X_wine_test_scaled = scaler_wine.transform(X_wine_test)

## Multiclass Strategies

1. **Softmax (Multinomial)**: Single model with softmax activation
2. **One-vs-Rest (OvR)**: Train binary classifier for each class

In [ ]:
# Strategy 1: Softmax
lr_softmax = LogisticRegression(max_iter=1000, random_state=42)
lr_softmax.fit(X_wine_train_scaled, y_wine_train)
y_pred_softmax = lr_softmax.predict(X_wine_test_scaled)
acc_softmax = accuracy_score(y_wine_test, y_pred_softmax)
print(f"Softmax Accuracy: {acc_softmax:.3f}")

# Strategy 2: One-vs-Rest
lr_ovr = LogisticRegression(max_iter=1000, random_state=42)
lr_ovr.fit(X_wine_train_scaled, y_wine_train)
y_pred_ovr = lr_ovr.predict(X_wine_test_scaled)
acc_ovr = accuracy_score(y_wine_test, y_pred_ovr)
print(f"One-vs-Rest Accuracy: {acc_ovr:.3f}")

## Key Takeaways

**Model Selection**:
1. AIC/BIC: Theoretical penalties for complexity
2. Cross-validation: Empirical evaluation

**Multiclass**:
1. Softmax preferred for most cases
2. OvR useful for interpretability